In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/OIA-MasterHPC/neuron
%ls pycore/
%ls config/
import importlib
import core.loader_neuronas

importlib.reload(core.loader_neuronas)

/content/drive/MyDrive/OIA-MasterHPC/neuron
cache_manager.py         macro_neurona.py           personalidad.py
clustering_conceptos.py  memoria.py                 priority_manager.py
config.py                MemoryNs.py                __pycache__/
consciencia.py           micro_neurona.py           razonador.py
grammar_adjudicator.py   MNs_01.py                  semantic_validator.py
hierarchical_memory.py   MNs_aprendidas.py          sintetizador_contexto.py
indices_vectoriales.py   neural_events.py           syntax_engine.py
lenguaje_utils.py        neurona_interconectora.py  TNs_01.py
loader_neuronas.py       neurona.py
neuro_config.yaml  personalidad.yaml


<module 'core.loader_neuronas' from '/content/drive/MyDrive/OIA-MasterHPC/neuron/core/loader_neuronas.py'>

In [ ]:
"""
Main para probar la arquitectura cognitiva con personalidad Aura.
Soporte multilingüe (español/inglés) y aprendizaje por refuerzo.

Uso:
    python main.py [--device cpu|cuda] [--config ruta_yaml] [--lang es|en] [--no-response]

Ejemplo:
    python main.py --device cpu --lang es
"""

import os
import sys
import argparse
import torch
import logging

from pycore.core.memory import Memoria
from pycore.core.personality import Personalidad
from pycore.core.cognitive_engine import CognitiveEngine
from pycore.loaders.neuron_loader import cargar_neuronas
from pycore.states.micro_state import ajustar_umbrales_micro, calcular_embedding
from pycore.states.neuron_state import ajustar_umbrales_neuronas
from pycore.learning.concept_clustering import GeneradorConceptos


def main():
    parser = argparse.ArgumentParser(description='Ejecutar Aura AI')
    parser.add_argument('--device', type=str, default='cpu', choices=['cpu', 'cuda'],
                        help='Dispositivo para PyTorch (cpu o cuda)')
    parser.add_argument('--config', type=str, default='config/personalidad.yaml',
                        help='Ruta al archivo YAML de personalidad')
    parser.add_argument('--lang', type=str, default='es', choices=['es', 'en'],
                        help='Idioma de la conversación (español o inglés)')
    parser.add_argument('--no-response', action='store_true',
                        help='No generar respuesta (solo mostrar activaciones)')
    parser.add_argument('--log-level', type=str, default='INFO',
                        choices=['DEBUG', 'INFO', 'WARNING', 'ERROR'],
                        help='Nivel de logging')
    args, unknown = parser.parse_known_args()

    # Configurar logging
    logging.basicConfig(
        level=getattr(logging, args.log_level),
        format='[%(levelname)s] %(name)s - %(message)s'
    )

    device = torch.device(args.device if torch.cuda.is_available() and args.device == 'cuda' else 'cpu')
    print(f"Usando dispositivo: {device}")
    print(f"Idioma seleccionado: {args.lang}")

    # 1. Cargar personalidad
    if not os.path.exists(args.config):
        print(f"Error: No se encuentra el archivo de personalidad {args.config}")
        sys.exit(1)
    personalidad = Personalidad(args.config)

    # 2. Crear memoria y razonador
    memoria = Memoria(device=device)
    razonador = CognitiveEngine(memoria, personalidad, device=device)

    # 3. Cargar neuronas (base, aprendidas, personalizadas)
    print("Cargando neuronas...")
    resumen = cargar_neuronas(razonador, device=device)
    print(f"Micro-neuronas: {len(resumen['micro_neuronas'])}")
    print(f"Neuronas: {len(resumen['neuronas'])}")
    print(f"Macro-neuronas: {len(resumen['macro_neuronas'])}")
    print(f"Interconectoras: {len(resumen['interconectoras'])}")

    if resumen['macro_neuronas']:
        print("Primeras macro‑neuronas:", resumen['macro_neuronas'][:5])

    # Inicializar generador de conceptos (clustering)
    try:
        generador_conceptos = GeneradorConceptos(
            razonador.micro_state,
            razonador.macro_state,
            umbral_coocurrencia=0.7,   # Podrías leerlo de configuración
            ventana=10
        )
        print("✅ Generador de conceptos inicializado")
    except Exception as e:
        print(f"❌ Error al inicializar generador de conceptos: {e}")
        generador_conceptos = None

    # 4. Poblar Thinking Neurons y MacroTN (si se quiere respuesta)
    if not args.no_response:
      razonador.inicializar_deliberacion(
          personalidad=personalidad,
          idioma=args.lang
      )
    else:
      macro_tn = None
      sintetizador = None
      conciencia = None

    # 5. Bucle conversacional
    print("\n" + "=" * 50)
    print("Sistema listo. Escribe 'salir' para terminar.")
    print("=" * 50)

    contador_interacciones = 0
    contador_clustering = 0

    while True:
        try:
            entrada = input("\nTú: ")
            if entrada.lower() in ('salir', 'exit', 'quit'):
                break

            # Generar embedding de la entrada
            if razonador.micro_state is not None and razonador.micro_state.n > 0:
                emb = calcular_embedding(entrada, dim=64).to(device)
                vectores_entrada = emb
            else:
                print("Error: No hay micro‑neuronas cargadas.")
                continue

            # Ejecutar ciclo de razonamiento (ya incluye deliberación si está inicializada)
            resultado = razonador.procesar_entrada_iterativo(
                vectores_entrada,
                frase_original=entrada,
                umbral_mn=0.3,
                num_iteraciones=10
            )

            # Extraer información del resultado
            estado_neuronal = resultado["estado_neuronal"]
            respuesta_generada = resultado.get("respuesta")
            plan_ganador = resultado.get("plan")

            # Depuración: mostrar macro‑neuronas activas
            if razonador.macro_state is not None:
                activas_macro = razonador.macro_state.activa.sum().item()
                if activas_macro > 0:
                    print(f"[DEBUG] Macro‑neuronas activas: {activas_macro}")

            # Extraer IDs activas para feedback y clustering
            micros_activas_ids = [mid for mid, act in estado_neuronal['micro_neuronas'].items() if act]
            neuronas_activas_ids = [nid for nid, act in estado_neuronal['neuronas'].items() if act]

            # Registrar en clustering
            if generador_conceptos:
                generador_conceptos.registrar_activaciones(micros_activas_ids)

            if args.no_response:
                # Modo solo activaciones
                print("\n[Activaciones finales]")
                print(f"Micro-neuronas activas: {len(micros_activas_ids)}")
                print(f"Neuronas activas: {len(neuronas_activas_ids)}")
                if micros_activas_ids:
                    print("IDs micro activas (primeras 10):", micros_activas_ids[:10])
            else:
                # Mostrar la respuesta generada por el razonador
                if respuesta_generada:
                    print(f"\nAura: {respuesta_generada}")
                else:
                    print("\nAura: No generé respuesta.")

                # --- Feedback del usuario (aprendizaje por refuerzo) ---
                feedback = input("¿Te ha parecido útil la respuesta? (s/n): ").lower()
                if feedback in ('s', 'n'):
                    exito = (feedback == 's')
                    razonador.reforzar_propuesta(
                        propuesta_exitosa=exito,
                        micros_activas_ids=micros_activas_ids,
                        neuronas_activas_ids=neuronas_activas_ids,
                        tasa_exito=0.05,
                        tasa_fracaso=-0.02
                    )
                    print("  (Refuerzo aplicado)" if exito else "  (Debilitamiento aplicado)")

                # --- Homeostasis periódica (cada 10 interacciones) ---
                contador_interacciones += 1
                if contador_interacciones % 10 == 0:
                    if razonador.micro_state:
                        razonador.micro_state = ajustar_umbrales_micro(razonador.micro_state)
                    if razonador.neurona_state:
                        razonador.neurona_state = ajustar_umbrales_neuronas(razonador.neurona_state)
                    print("Homeostasis aplicada")
                    if razonador.micro_state:
                        print(f"    Umbral micro medio: {razonador.micro_state.umbral_activacion.mean().item():.3f}")
                    if razonador.neurona_state:
                        print(f"    Umbral neurona medio: {razonador.neurona_state.umbral_activacion.mean().item():.3f}")

            # --- Clustering: cada 20 interacciones, buscar nuevos conceptos ---
            contador_clustering += 1
            if generador_conceptos and contador_clustering % 20 == 0:
                print("\n[CLUSTERING] Buscando nuevos conceptos...")
                nuevos = generador_conceptos.detectar_nuevos_conceptos()
                if nuevos:
                    print(f"[CLUSTERING] Se han detectado {len(nuevos)} posibles nuevos conceptos.")
                for grupo, conf in nuevos:
                    nuevo_id = generador_conceptos.crear_nuevo_concepto(grupo, conf, razonador)

                    if nuevo_id and razonador.conciencia is not None:
                        razonador.conciencia.add_concept(nuevo_id, list(grupo))
                        print(f"[CLUSTERING] Concepto {nuevo_id} añadido a la conciencia.")

                if nuevos:
                    print("[CLUSTERING] Proceso completado.\n")

        except KeyboardInterrupt:
            print("\nInterrupción detectada.")
            break
        except Exception as e:
            print(f"\nError: {e}")
            import traceback
            traceback.print_exc()

    print("Sistema finalizado.")


if __name__ == '__main__':
    main()

Usando dispositivo: cpu
Idioma seleccionado: es
Cargando neuronas...
Micro-neuronas: 193
Neuronas: 15
Macro-neuronas: 33
Interconectoras: 0
Primeras macro‑neuronas: ['concepto_saludo', 'concepto_despedida', 'concepto_pregunta_bienestar', 'concepto_respuesta_bienestar', 'concepto_pregunta_reciproca']
✅ Generador de conceptos inicializado
[DEBUG] Pobladas 3 neuronas de pensamiento especializado.

Sistema listo. Escribe 'salir' para terminar.
[DEBUG Sintetizador] Generando hipótesis con 4 neuronas activas y 0 micros activas
[DEBUG Sintetizador] Hipótesis añadida: tipo=neuronal_focus, elementos=['n_peticion_ayuda']
[DEBUG Sintetizador] Hipótesis añadida (cálculo): tipo=pregunta_calculo, elementos=['n_pregunta_calculo_simple']
[DEBUG Sintetizador] Hipótesis añadida (cálculo): tipo=pregunta_calculo, elementos=['n_pregunta_calculo_ingles']
[DEBUG Sintetizador] Hipótesis añadida: tipo=thinking_memory_focus, elementos=['mn_adios']
[DEBUG Sintetizador] Hipótesis añadida: tipo=thinking_memory_foc

[DEBUG Sintetizador] Generando hipótesis con 4 neuronas activas y 0 micros activas
[DEBUG Sintetizador] Hipótesis añadida: tipo=neuronal_focus, elementos=['n_peticion_ayuda']
[DEBUG Sintetizador] Hipótesis añadida (cálculo): tipo=pregunta_calculo, elementos=['n_pregunta_calculo_simple']
[DEBUG Sintetizador] Hipótesis añadida (cálculo): tipo=pregunta_calculo, elementos=['n_pregunta_calculo_ingles']
[DEBUG Sintetizador] Hipótesis añadida: tipo=thinking_memory_focus, elementos=['mn_see_you']
[DEBUG Sintetizador] Hipótesis añadida: tipo=thinking_memory_focus, elementos=['gen_and_you']
[DEBUG Sintetizador] Hipótesis añadida: tipo=thinking_memory_focus, elementos=['mn_you']
[DEBUG Sintetizador] Hipótesis añadida: tipo=thinking_memory_focus, elementos=['mn_estoy']
[DEBUG Sintetizador] Hipótesis añadida: tipo=thinking_memory_focus, elementos=['gen_help_you']
[DEBUG Sintetizador] Total hipótesis generadas: 8
[DEBUG MacroTN] Iniciando ciclo con 8 hipótesis de contexto
[DEBUG TN ProtocoloSocial] 

[DEBUG Sintetizador] Generando hipótesis con 3 neuronas activas y 0 micros activas
[DEBUG Sintetizador] Hipótesis añadida: tipo=neuronal_focus, elementos=['n_peticion_nombre']
[DEBUG Sintetizador] Hipótesis añadida (cálculo): tipo=pregunta_calculo, elementos=['n_pregunta_calculo_simple']
[DEBUG Sintetizador] Hipótesis añadida (cálculo): tipo=pregunta_calculo, elementos=['n_pregunta_calculo_ingles']
[DEBUG Sintetizador] Hipótesis añadida: tipo=thinking_memory_focus, elementos=['mn_yo']
[DEBUG Sintetizador] Hipótesis añadida: tipo=thinking_memory_focus, elementos=['mn_i']
[DEBUG Sintetizador] Total hipótesis generadas: 5
[DEBUG MacroTN] Iniciando ciclo con 5 hipótesis de contexto
[DEBUG TN ProtocoloSocial] Evaluando 5 hipótesis
[DEBUG TN ProtocoloSocial] Hipótesis 0: tipo=neuronal_focus, subtipo=None, confianza=1.0
[DEBUG TN ProtocoloSocial] Hipótesis 1: tipo=pregunta_calculo, subtipo=None, confianza=0.7499999999999998
[DEBUG TN ProtocoloSocial] Hipótesis 2: tipo=pregunta_calculo, subtip

[DEBUG Sintetizador] Generando hipótesis con 3 neuronas activas y 0 micros activas
[DEBUG Sintetizador] Hipótesis añadida: tipo=neuronal_focus, elementos=['n_peticion_nombre']
[DEBUG Sintetizador] Hipótesis añadida (cálculo): tipo=pregunta_calculo, elementos=['n_pregunta_calculo_simple']
[DEBUG Sintetizador] Hipótesis añadida (cálculo): tipo=pregunta_calculo, elementos=['n_pregunta_calculo_ingles']
[DEBUG Sintetizador] Hipótesis añadida: tipo=thinking_memory_focus, elementos=['gen_and_you']
[DEBUG Sintetizador] Hipótesis añadida: tipo=thinking_memory_focus, elementos=['mn_ok']
[DEBUG Sintetizador] Hipótesis añadida: tipo=thinking_memory_focus, elementos=['mn_see_you']
[DEBUG Sintetizador] Hipótesis añadida: tipo=thinking_memory_focus, elementos=['mn_chao']
[DEBUG Sintetizador] Hipótesis añadida: tipo=thinking_memory_focus, elementos=['gen_well']
[DEBUG Sintetizador] Total hipótesis generadas: 8
[DEBUG MacroTN] Iniciando ciclo con 8 hipótesis de contexto
[DEBUG TN ProtocoloSocial] Evalu

[DEBUG Sintetizador] Generando hipótesis con 1 neuronas activas y 0 micros activas
[DEBUG Sintetizador] Hipótesis añadida: tipo=neuronal_focus, elementos=['n_pregunta_calculo_simple']
[DEBUG Sintetizador] Hipótesis añadida (cálculo): tipo=pregunta_calculo, elementos=['n_pregunta_calculo_simple']
[DEBUG Sintetizador] Hipótesis añadida: tipo=thinking_memory_focus, elementos=['mn_seven']
[DEBUG Sintetizador] Hipótesis añadida: tipo=thinking_memory_focus, elementos=['mn_see_you']
[DEBUG Sintetizador] Hipótesis añadida: tipo=thinking_memory_focus, elementos=['mn_noches']
[DEBUG Sintetizador] Hipótesis añadida: tipo=thinking_memory_focus, elementos=['gen_bien']
[DEBUG Sintetizador] Hipótesis añadida: tipo=thinking_memory_focus, elementos=['mn_weather']
[DEBUG Sintetizador] Total hipótesis generadas: 7
[DEBUG MacroTN] Iniciando ciclo con 7 hipótesis de contexto
[DEBUG TN ProtocoloSocial] Evaluando 7 hipótesis
[DEBUG TN ProtocoloSocial] Hipótesis 0: tipo=neuronal_focus, subtipo=None, confianza